In [0]:
# Create struct schema for suppliers csv file

from pyspark.sql.types import *

suppliers_schema = StructType([
    StructField("supplier_id", IntegerType(), True),
    StructField("country", StringType(), True)
])

In [0]:
# autoload csv into dataframe with schema location defined

df = pyspark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option(
        "cloudFiles.schemaLocation", 
        "/Volumes/first_data_engineering_project/pipeline_metadata/autoloader_metadata/schemas/bronze/bronze_suppliers/"
        ) \
    .schema(suppliers_schema) \
    .load("/Volumes/first_data_engineering_project/landing/retail_files/suppliers/")

In [0]:
# add bronze layer metadata columns to the dataFrame for ingestion and lineage tracking

from pyspark.sql import functions as F

bronze_suppliers = df \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("source_file", F.col("_metadata.file_name")) \
    .withColumn("file_modified_time", F.col("_metadata.file_modification_time"))

In [0]:
# create table with checkpoint location and add trigger

bronze_suppliers.writeStream \
    .option(
        "checkpointLocation", 
        "/Volumes/first_data_engineering_project/pipeline_metadata/autoloader_metadata/checkpoints/bronze/bronze_suppliers/"
        ) \
    .trigger(availableNow=True) \
    .toTable("first_data_engineering_project.bronze.bronze_suppliers")